# **Cross-Family LLM Stitching: Comparing Reconstruction and Prediction-Based Objectives**

*COMS 4705 - Natural Language Processing - Final Project*

### **Authors**

Caroline Zhang (cz2977@columbia.edu)

Siddharth Grover (sg4792@columbia.edu)

Edwin Argudo (eaa2193@columbia.edu)

### **GitHub Repo**

https://github.com/edwin-a14/coms-4705-project-cross-model-stitching


## **Milestone 1**

**Prerequisite:** Run the **entire [smoke-test.ipynb](https://github.com/edwin-a14/coms-4705-project-cross-model-stitching/blob/306fe3c3fdd168b53780c51a42c786d304c991c8/notebooks/smoke-test.ipynb) successfully**
before using this notebook to verify the environment, model access, activation extraction,
output-head reconstruction, cache loading, and the small affine MSE example.

This notebook organizes the data preparation, cached representations, baseline comparisons,
and validation protocol for cross-family LLM stitching. The experiment maps Llama-3.2-3B's
final residual representation to Qwen2.5-3B's representation space using an affine transformation.
Milestone 1 establishes the data and evaluation framework for comparing reconstruction and
prediction objectives in Milestone 2.

Dataset construction is defined in [dataset.ipynb](https://github.com/edwin-a14/coms-4705-project-cross-model-stitching/blob/306fe3c3fdd168b53780c51a42c786d304c991c8/notebooks/dataset.ipynb), and activation extraction
is defined in [activation.ipynb](https://github.com/edwin-a14/coms-4705-project-cross-model-stitching/blob/306fe3c3fdd168b53780c51a42c786d304c991c8/notebooks/activation.ipynb). This notebook uses their saved artifacts
for analysis, with the selected runs identified by manifest checksums and model/tokenizer revisions.

### **Deliverables**

| Phase | Deliverable |
|---|---|
| 1. LAMBADA preprocessing and manifests | Reproducible example manifests, aligned labels, and a documented sampling protocol |
| 2. Sequential activation extraction and persistent caching | Paired activation caches, output-component specifications, and extraction metadata |
| 3. Native and constant-activation baselines | Native Llama, native Qwen, and train-mean Qwen baseline comparisons with sample counts |
| 4. Pipeline validation | Data alignment, split separation, reconstruction, and numerical consistency checks |
| 5. Prep for MSE/CE/KL experiments | Documented cache interface, baseline summary, and an untrained affine scaffold |

**Evaluation scope:** The primary task is prediction of the first continuation token. Full-word
LAMBADA evaluation requires a separate protocol. Llama's token-level scores are reported in its
own vocabulary, while the constant baseline and stitched-model evaluation use Qwen's vocabulary.
Analysis in this notebook focuses on training and validation; test scores are reserved for final evaluation.


## **Dependencies & Configuration**


In [1]:
%pip install -q \
    "transformers>=5,<6" \
    "huggingface_hub>=1.32,<2" \
    "datasets>=3,<5" \
    "accelerate>=1.0" \
    safetensors \
    tqdm \
    "nltk>=3.9.1,<4" \
    numpy \
    matplotlib

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/coms-4705-project')
TRAIN_DIR = Path(
    "/content/drive/MyDrive/coms-4705-project/datasets/lambada/train/7b6847a7f51353d0"
)
CACHE_RUN_DIR = Path(
    "/content/drive/MyDrive/coms-4705-project/activations/lambada/"
    "7b6847a7f51353d0/train10000_validation4869_a938da615918"
)
ANALYSIS_SPLITS = ('train', 'validation')

## **Phase 1 — LAMBADA Preprocessing and Manifests**

**Implementation:** [dataset.ipynb](https://github.com/edwin-a14/coms-4705-project-cross-model-stitching/blob/306fe3c3fdd168b53780c51a42c786d304c991c8/notebooks/dataset.ipynb)

LAMBADA provides training novels and separate validation/test passages. Training examples are
sampled from the novels as context-and-next-word pairs, with passage lengths drawn from the
validation length distribution. The sampler selects noun targets that also occur earlier in
the context, normalizes training whitespace to single spaces, and distributes sampling across
novels. These criteria define the training distribution used for the stitching experiment.

Each example uses the same text prefix for both models and must fit within 256 tokens under
both tokenizers. Labels are derived by tokenizing the complete prefix and continuation, checking
that the prefix token IDs remain unchanged, and selecting the first continuation token. Multi-token
targets are retained and identified separately for subset evaluation.

The training manifest stores example IDs, source locations, prefixes, continuations, and model-specific
labels. Its accompanying provenance file records the dataset/model revisions, sampling settings,
and manifest checksum. This section characterizes the selected data through sample counts,
passage and token lengths and target characteristics. Split comparisons in Phase 2
cover the full training manifest and the selected validation cache manifest.


In [6]:
import hashlib
import json
from collections import Counter
from itertools import combinations
from IPython.display import Markdown, display


def show_table(title, columns, rows):
    def format_value(value):
        if isinstance(value, float):
            return f"{value:.3f}"
        return str(value).replace("|", "\\|").replace("\n", " ")
    lines = [f"### {title}", "", "| " + " | ".join(columns) + " |",
             "| " + " | ".join("---" for _ in columns) + " |"]
    lines += ["| " + " | ".join(map(format_value, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def read_manifest(path, expected_sha256):
    data = Path(path).read_bytes()
    if sha256_bytes(data) != expected_sha256:
        raise ValueError(f"Manifest checksum mismatch: {path}")
    rows = []
    for line_number, line in enumerate(data.splitlines(), 1):
        if not line.strip():
            raise ValueError(f"Blank manifest row at {path}:{line_number}")
        row = json.loads(line)
        if not isinstance(row, dict):
            raise ValueError(f"Expected a JSON object at {path}:{line_number}")
        rows.append(row)
    if not rows:
        raise ValueError(f"Empty manifest: {path}")
    return rows


def normalized_context(row):
    return " ".join(row["prefix"].split())


def validate_manifest(rows, split, max_tokens):
    ids, contexts = set(), set()
    for row in rows:
        if row.get("split") != split or not isinstance(row.get("id"), str):
            raise ValueError(f"Invalid split/ID in {split} manifest")
        for key in ("prefix", "continuation"):
            if not isinstance(row.get(key), str) or not row[key].strip():
                raise ValueError(f"Invalid {key} for {row['id']}")
        if row["id"] in ids:
            raise ValueError(f"Duplicate ID in {split}: {row['id']}")

        if split == "train" and normalized_context(row) in contexts:
            raise ValueError(
                f"Duplicate normalized training context: {row['id']}"
            )
        ids.add(row["id"])
        contexts.add(normalized_context(row))
        if row.get("n_words") != len((row["prefix"] + row["continuation"]).split()):
            raise ValueError(f"Word count mismatch for {row['id']}")
        for name in ("llama", "qwen"):
            for suffix in ("length", "target_id", "target_length"):
                value = row.get(f"{name}_{suffix}")
                if type(value) is not int or value < (0 if suffix == "target_id" else 1):
                    raise ValueError(f"Invalid {name}_{suffix} for {row['id']}")
            if row[f"{name}_length"] > max_tokens:
                raise ValueError(f"Context cap exceeded for {row['id']}")


def load_training_artifacts(directory):
    if directory is None:
        raise ValueError("Set TRAIN_DIR to the saved training directory printed by dataset.ipynb.")
    directory = Path(directory)
    meta = json.loads((directory / "provenance.json").read_text())
    if meta.get("split") != "train" or set(meta.get("models", {})) != {"llama", "qwen"}:
        raise ValueError("Expected training provenance for Llama and Qwen")
    if set(meta.get("revisions", {})) != {"llama", "qwen"}:
        raise ValueError("Missing tokenizer/model revisions")
    rows = read_manifest(directory / "examples.jsonl", meta["manifest_sha256"])
    if len(rows) != meta["n_train"]:
        raise ValueError("Training row count differs from provenance")
    validate_manifest(rows, "train", meta["max_tokens"])
    for row in rows:
        if len(row["id"].split("/")) != 4 or not row["id"].startswith("train/"):
            raise ValueError("Expected train/<novel>/<start>/<prefix_end> source IDs")
        if row.get("target_pos") not in meta["target_pos_tags"]:
            raise ValueError(f"Target POS violates saved sampling policy: {row['id']}")
        if row["continuation"].strip().casefold() not in {word.casefold() for word in row["prefix"].split()}:
            raise ValueError(f"Training target does not occur in its prefix: {row['id']}")
    return rows, meta


def describe_lengths(values):
    values = sorted(values)
    def percentile(q):
        return values[round(q * (len(values) - 1))]
    return [len(values), min(values), sum(values)/len(values), percentile(.5), percentile(.95), max(values)]


training_rows, training_meta = load_training_artifacts(TRAIN_DIR)
training_by_id = {row["id"]: row for row in training_rows}
dataset_summary = {
    "examples": len(training_rows),
    "source_novels": len({row["id"].split("/")[1] for row in training_rows}),
    "manifest_sha256": training_meta["manifest_sha256"],
    "target_pos_counts": dict(Counter(row["target_pos"] for row in training_rows)),
}
show_table("Training data", ["Measure", "Value"], [
    ["Examples", dataset_summary["examples"]], ["Source novels", dataset_summary["source_novels"]],
    ["Dataset", training_meta["dataset"]], ["Seed", training_meta["seed"]],
    ["Context cap (tokens/model)", training_meta["max_tokens"]],
    ["Target POS tags", ", ".join(training_meta["target_pos_tags"])],
    ["Target occurrence rule", training_meta["target_in_prefix"]],
    ["Whitespace policy", training_meta["whitespace"]], ["Sampling policy", training_meta["sampling"]],
])
show_table("Passage and context lengths", ["Measure", "N", "Min", "Mean", "Median", "P95", "Max"],
           [[label, *describe_lengths(row[field] for row in training_rows)] for label, field in
            [("Passage words (including target)", "n_words"), ("Llama prefix tokens", "llama_length"),
             ("Qwen prefix tokens", "qwen_length")]])
show_table("Target tokenization", ["Model", "Single-token targets", "All targets", "Single-token fraction"],
           [[name, sum(row[f"{name}_target_length"] == 1 for row in training_rows), len(training_rows),
             sum(row[f"{name}_target_length"] == 1 for row in training_rows)/len(training_rows)]
            for name in ("llama", "qwen")])
show_table("Target POS distribution", ["Tag", "Examples"], sorted(dataset_summary["target_pos_counts"].items()))
show_table("Artifact provenance", ["Field", "Value"], [
    ["Training directory", Path(TRAIN_DIR)], ["Manifest SHA256", training_meta["manifest_sha256"]],
    ["Dataset revision", training_meta["dataset_revision"]],
    *[[f"{name} model", training_meta["models"][name]] for name in ("llama", "qwen")],
    *[[f"{name} revision", training_meta["revisions"][name]] for name in ("llama", "qwen")],
])


### Training data

| Measure | Value |
| --- | --- |
| Examples | 10000 |
| Source novels | 2500 |
| Dataset | cimec/lambada |
| Seed | 42 |
| Context cap (tokens/model) | 256 |
| Target POS tags | NN, NNP, NNPS, NNS |
| Target occurrence rule | case-insensitive whitespace-delimited word |
| Whitespace policy | single spaces |
| Sampling policy | balanced first pass with productive-novel refill |

### Passage and context lengths

| Measure | N | Min | Mean | Median | P95 | Max |
| --- | --- | --- | --- | --- | --- | --- |
| Passage words (including target) | 10000 | 59 | 74.974 | 73 | 95 | 170 |
| Llama prefix tokens | 10000 | 59 | 81.627 | 79 | 105 | 195 |
| Qwen prefix tokens | 10000 | 58 | 80.715 | 78 | 104 | 194 |

### Target tokenization

| Model | Single-token targets | All targets | Single-token fraction |
| --- | --- | --- | --- |
| llama | 7740 | 10000 | 0.774 |
| qwen | 7728 | 10000 | 0.773 |

### Target POS distribution

| Tag | Examples |
| --- | --- |
| NN | 8917 |
| NNP | 7 |
| NNPS | 1 |
| NNS | 1075 |

### Artifact provenance

| Field | Value |
| --- | --- |
| Training directory | /content/drive/MyDrive/coms-4705-project/datasets/lambada/train/7b6847a7f51353d0 |
| Manifest SHA256 | 604387bf620dfe540501006eb7977fa73c99a31454f9e6093c0cc497747d3044 |
| Dataset revision | 5953bd97664b64b95754f299b2309ecfbfbe81b9 |
| llama model | meta-llama/Llama-3.2-3B |
| qwen model | Qwen/Qwen2.5-3B |
| llama revision | 13afe5124825b4f3751f836b40dafda64c1ed062 |
| qwen revision | 3aab1f1954e9cc14eb9509a215f9e5ca08227a9b |

## **Phase 2 — Sequential Activation Extraction and Persistent Caching**

**Implementation:** [activation.ipynb](https://github.com/edwin-a14/coms-4705-project-cross-model-stitching/blob/306fe3c3fdd168b53780c51a42c786d304c991c8/notebooks/activation.ipynb)

Activation extraction pairs representations of identical text contexts from Llama and Qwen.
The models are loaded sequentially to limit GPU memory use. For each context, a hook at the
final RMSNorm layer captures the residual vector before normalization and the corresponding
normalized vector. With right-padded batches, the attention-mask length identifies the last
real context token for each example.

The pre-RMSNorm vectors are the source and target representations for stitching: 3,072 dimensions
for Llama and 2,048 for Qwen. The post-RMSNorm vectors support normalization checks. Saved split
artifacts contain example manifests, per-model tensors, and a combined `paired.pt` payload with
ordered IDs, labels, representations, and provenance. The keys `h_llama` and `h_qwen` refer to
pre-RMSNorm vectors; the `_post` suffix identifies normalized representations.

This section describes the selected cache's schema, sample counts, tensor dimensions, and
precision, and connects each row to its manifest example. The final normalization and LM-head
parameters define the output computation used for native baselines and later prediction-based
objectives. Their reconstruction accuracy is evaluated in Phase 4.


In [7]:
import torch

HIDDEN_SIZES = {"llama": 3072, "qwen": 2048}


def load_paired_artifacts(directory, split, parent_rows, parent_meta):
    if split not in ("train", "validation"):
        raise ValueError("Artifact analysis is restricted to train and validation.")
    if directory is None:
        raise ValueError("Set CACHE_RUN_DIR to the saved extraction run printed by activation.ipynb.")
    folder = Path(directory) / split
    cache = torch.load(folder / "paired.pt", map_location="cpu", weights_only=True)
    meta = cache["provenance"]
    for key in ("dataset", "dataset_revision", "models", "revisions", "max_tokens"):
        if meta.get(key) != parent_meta[key]:
            raise ValueError(f"{split} cache differs from training provenance: {key}")
    if meta.get("split") != split:
        raise ValueError(f"Cache split mismatch: {split}")
    if cache.get("source_model") != parent_meta["models"]["llama"] or cache.get("target_model") != parent_meta["models"]["qwen"]:
        raise ValueError("Paired-cache model IDs do not match the selected dataset")
    rows = read_manifest(folder / "examples.jsonl", meta["manifest_sha256"])
    validate_manifest(rows, split, meta["max_tokens"])
    if len(rows) != meta["n_examples"] or cache["ids"] != [row["id"] for row in rows]:
        raise ValueError(f"Cache row count/ID order mismatch: {split}")
    if split == "train":
        parent_by_id = {row["id"]: row for row in parent_rows}
        if any(parent_by_id.get(row["id"]) != row for row in rows):
            raise ValueError("Cached training rows are not an exact subset of the selected training manifest")
    for name in ("llama", "qwen"):
        label_key = "target_token_ids" if name == "qwen" else "llama_target_token_ids"
        expected = torch.tensor([row[f"{name}_target_id"] for row in rows], dtype=torch.int64)
        labels = cache[label_key]
        if not isinstance(labels, torch.Tensor) or labels.dtype != torch.int64 or not torch.equal(labels, expected):
            raise ValueError(f"Cached {name} labels do not match manifest order/content")
        for suffix in ("", "_post"):
            key = f"h_{name}{suffix}"
            tensor = cache[key]
            if not isinstance(tensor, torch.Tensor) or tensor.shape != (len(rows), HIDDEN_SIZES[name]):
                raise ValueError(f"Invalid tensor shape: {split}/{key}")
            if tensor.device.type != "cpu" or not tensor.is_floating_point() or not torch.isfinite(tensor).all():
                raise ValueError(f"Invalid device/dtype/nonfinite activation: {split}/{key}")
    return rows, cache


def check_split_overlap(manifests):
    results = []
    for (left, left_rows), (right, right_rows) in combinations(manifests.items(), 2):
        ids = {row["id"] for row in left_rows} & {row["id"] for row in right_rows}
        contexts = {normalized_context(row) for row in left_rows} & {normalized_context(row) for row in right_rows}
        results.append([left, len(left_rows), right, len(right_rows), len(ids), len(contexts)])
        if ids or contexts:
            raise ValueError(f"Split overlap between {left} and {right}: {len(ids)} IDs, {len(contexts)} normalized contexts")
    return results


if not ANALYSIS_SPLITS or len(set(ANALYSIS_SPLITS)) != len(ANALYSIS_SPLITS) or any(s not in ("train", "validation") for s in ANALYSIS_SPLITS):
    raise ValueError("Select unique analysis splits from train and validation only.")
loaded = {split: load_paired_artifacts(CACHE_RUN_DIR, split, training_rows, training_meta)
          for split in ANALYSIS_SPLITS}
cache_manifests = {split: value[0] for split, value in loaded.items()}
paired_caches = {split: value[1] for split, value in loaded.items()}
overlap_rows = check_split_overlap({"Full training manifest": training_rows,
                                   **{f"Selected {s} manifest": rows for s, rows in cache_manifests.items() if s != "train"}})
cache_summary = []
for split, cache in paired_caches.items():
    for name in ("llama", "qwen"):
        for suffix, stage in (("", "pre-RMSNorm"), ("_post", "post-RMSNorm")):
            tensor = cache[f"h_{name}{suffix}"]
            cache_summary.append([split, name, stage, tensor.shape[0], tensor.shape[1], str(tensor.dtype),
                                  tensor.numel()*tensor.element_size()/1024**2,
                                  tensor.float().norm(dim=-1).mean().item()])
show_table("Cached representations", ["Split", "Model", "Stage", "Rows", "Width", "Dtype", "MiB", "Mean L2 norm"], cache_summary)
show_table("Cache sample coverage", ["Split", "Cached examples", "Available training examples"],
           [[split, len(cache_manifests[split]), len(training_rows) if split == "train" else "—"]
            for split in ANALYSIS_SPLITS])
show_table("Selected-manifest overlap", ["Manifest A", "Rows A", "Manifest B", "Rows B", "Shared IDs", "Shared contexts"], overlap_rows)
show_table("Cache provenance", ["Split", "Directory", "Manifest SHA256"],
           [[split, Path(CACHE_RUN_DIR)/split, paired_caches[split]["provenance"]["manifest_sha256"]]
            for split in ANALYSIS_SPLITS])
display(Markdown("Overlap comparisons cover only the manifests and sample counts shown above; "
                 "they do not establish source-book separation. The `_post` tensors are normalized "
                 "representations, not normalization parameters or output-head weights."))


### Cached representations

| Split | Model | Stage | Rows | Width | Dtype | MiB | Mean L2 norm |
| --- | --- | --- | --- | --- | --- | --- | --- |
| train | llama | pre-RMSNorm | 10000 | 3072 | torch.bfloat16 | 58.594 | 63.066 |
| train | llama | post-RMSNorm | 10000 | 3072 | torch.bfloat16 | 58.594 | 87.851 |
| train | qwen | pre-RMSNorm | 10000 | 2048 | torch.bfloat16 | 39.062 | 553.300 |
| train | qwen | post-RMSNorm | 10000 | 2048 | torch.bfloat16 | 39.062 | 171.531 |
| validation | llama | pre-RMSNorm | 4869 | 3072 | torch.bfloat16 | 28.529 | 66.666 |
| validation | llama | post-RMSNorm | 4869 | 3072 | torch.bfloat16 | 28.529 | 87.369 |
| validation | qwen | pre-RMSNorm | 4869 | 2048 | torch.bfloat16 | 19.020 | 585.719 |
| validation | qwen | post-RMSNorm | 4869 | 2048 | torch.bfloat16 | 19.020 | 165.830 |

### Cache sample coverage

| Split | Cached examples | Available training examples |
| --- | --- | --- |
| train | 10000 | 10000 |
| validation | 4869 | — |

### Selected-manifest overlap

| Manifest A | Rows A | Manifest B | Rows B | Shared IDs | Shared contexts |
| --- | --- | --- | --- | --- | --- |
| Full training manifest | 10000 | Selected validation manifest | 4869 | 0 | 0 |

### Cache provenance

| Split | Directory | Manifest SHA256 |
| --- | --- | --- |
| train | /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/train | 604387bf620dfe540501006eb7977fa73c99a31454f9e6093c0cc497747d3044 |
| validation | /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/validation | d0d9208aab32b2b16c40368c538b654a00589e9ea834b1b7e5cb96824e3f5753 |

Overlap comparisons cover only the manifests and sample counts shown above; they do not establish source-book separation. The `_post` tensors are normalized representations, not normalization parameters or output-head weights.

## **Phase 3 — Native and Constant-Activation Baselines**

The baseline comparisons establish reference performance before fitting an affine stitch.
Native Llama and native Qwen use their own cached representations and corresponding normalization
and output heads. The constant-activation baseline predicts Qwen's mean training representation
for every context, then applies Qwen's normalization and output head. Only training examples
contribute to the mean.

Evaluation reports next-token cross-entropy and top-1 accuracy, along with accuracy and sample
counts for targets represented by a single token. Training and validation results are presented
separately. Llama's scores use its own tokenizer and are interpreted as a source-model reference;
agreement and KL comparisons with Qwen are computed within Qwen's vocabulary. Every metric is
associated with the selected manifest and its evaluation denominator.


In [ ]:
# TODO: Load verified output components and compute the mean from the selected Qwen training cache.
# TODO: Record the mean's training sample count and manifest checksum.
# TODO: Evaluate native Qwen, native Llama, and constant Qwen representations on train/validation.
# TODO: Present metrics with sample counts, tokenizer names, and artifact provenance.


## **Phase 4 — Pipeline Validation**

Validation checks the correspondence between text, labels, cached representations, and reported
scores. Data checks cover source IDs, split overlap, token boundaries, label ranges, manifest
checksums, and paired row order. Tensor checks cover dimensions, precision, finite values, and
equality after saving and reloading.

Model checks compare batched extraction with single-example extraction and verify that applying
the saved normalization and output head to a pre-RMSNorm representation reproduces the original
model's logits within a stated numerical tolerance. Metric checks independently recompute a small
batch and verify that the constant representation uses training data only. The analysis records
the example IDs, observed differences, tolerances, and artifact references used in each comparison.


In [ ]:
# TODO: Check split/label alignment, cache integrity, and train-only mean provenance.
# TODO: Compare batched and single-example activations using stated numerical tolerances.
# TODO: Compare reconstructed vocabulary logits with independently obtained native logits.
# TODO: Recompute a small metric batch and verify cache loading in a fresh runtime.
# TODO: Present validation measurements, tolerances, and evidence for the selected artifact run.


## **Phase 5 — Prep for MSE/CE/KL experiments**

The Milestone 2 interface connects paired activation caches to an affine transformation from
Llama's 3,072-dimensional representation to Qwen's 2,048-dimensional representation. A model-free
loader supplies aligned source activations, target activations, Qwen labels, and example IDs.
The affine scaffold specifies the mapping and initialization without training it in this notebook.

MSE compares predicted and native Qwen pre-RMSNorm activations. CE evaluates the predicted
representation through Qwen's frozen normalization and output head against the ground-truth
Qwen label. KL compares the resulting distribution with native Qwen's distribution, which can
be recomputed batchwise from cached Qwen activations through the same output components.

This section documents those interfaces and summarizes the dataset, cache schema, baseline
comparisons, and validation results used for the objective comparison. It also identifies the
exact artifacts and split definitions so the experiments share a consistent evaluation framework.


In [ ]:
# TODO: Specify a model-free loader yielding float32 X/Y, Qwen labels, and stable IDs by split.
# TODO: Prepare an untrained nn.Linear(3072, 2048) scaffold and document objective inputs.
# TODO: Describe frozen-head gradient flow and batchwise native-Qwen teacher computation for CE/KL.
# TODO: Summarize dataset/cache provenance, baseline comparisons, and validation results.
